# Example: Building a Classical Minimum-Variance Portfolio

In this example, we generate synthetic asset return data, estimate the expected return vector and covariance matrix, solve the minimum-variance optimization problem, compute the efficient frontier, and explore how sensitive the solution is to estimation error.

> __Learning Objectives:__
>
> * **Synthetic Data Generation**: Build a controlled multi-asset return dataset with known ground-truth parameters using a multivariate normal model
> * **Statistical Estimation**: Estimate $\boldsymbol{\mu}$ and $\boldsymbol{\Sigma}$ from sample data and quantify estimation error relative to the true parameters
> * **Quadratic Optimization**: Solve the Markowitz minimum-variance portfolio problem using JuMP and trace out the efficient frontier
> * **Sensitivity Analysis**: Demonstrate the "error maximization" property by perturbing expected returns and observing weight instability

## Setup, Data and Prerequisites
We begin by loading our packages and helper functions via the `Include.jl` file. This activates the local Julia environment and loads all dependencies.

In [ ]:
include("Include.jl");

___
## Task 1: Define Our Asset Universe and Generate Synthetic Data
We'll work with a universe of $N = 5$ synthetic assets, loosely inspired by a diversified portfolio: a large-cap equity, a small-cap equity, an international equity, a bond, and a commodity. We define "true" parameters for each asset and use a multivariate normal distribution to generate synthetic daily returns.

> __What are we going to do?__ We'll build the ground-truth parameters step by step: first the expected returns and volatilities, then the correlation structure, then the full covariance matrix $\boldsymbol{\Sigma} = \mathbf{D}\mathbf{C}\mathbf{D}$ where $\mathbf{D} = \text{diag}(\sigma_1,\ldots,\sigma_N)$. Having the true parameters lets us measure _exactly_ how estimation error affects the optimizer.
>
> __Why synthetic data?__ In later sessions, we'll use [JumpHMM.jl](https://github.com/varnerlab/JumpHMM.jl) to generate more realistic regime-switching synthetic paths. For now, the multivariate normal gives us a clean baseline.

In [ ]:
# initialize -
asset_names = ["LargeCap", "SmallCap", "International", "Bond", "Commodity"];
N = length(asset_names);

# "True" annualized expected returns and volatilities -
μ_annual = [0.10, 0.12, 0.08, 0.03, 0.06];  # annual returns
σ_annual = [0.18, 0.25, 0.22, 0.05, 0.20];   # annual volatilities

# convert to daily -
μ_true = μ_annual ./ 252;
σ_daily = σ_annual ./ sqrt(252);

# display -
params_df = DataFrame(
    "Asset" => asset_names,
    "μ (annual %)" => μ_annual .* 100,
    "σ (annual %)" => σ_annual .* 100,
    "μ (daily bps)" => round.(μ_true .* 10000, digits=2),
    "σ (daily bps)" => round.(σ_daily .* 10000, digits=2)
);
pretty_table(params_df, tf = tf_markdown)

Next, we define the "true" correlation matrix $\mathbf{C}$. Notice the structure: equities are positively correlated with each other (0.55–0.70), the bond is _negatively_ correlated with equities (−0.05 to −0.10), and the commodity has low positive correlations with everything. This is a stylized version of the diversification structure investors rely on.

In [ ]:
# "True" correlation matrix -
C_true = [
    1.00  0.70  0.65  -0.10  0.15;
    0.70  1.00  0.55  -0.05  0.20;
    0.65  0.55  1.00  -0.08  0.25;
   -0.10 -0.05 -0.08   1.00 -0.05;
    0.15  0.20  0.25  -0.05  1.00
];

# visualize -
heatmap(C_true, xticks=(1:N, asset_names), yticks=(1:N, asset_names),
    title="True Correlation Matrix", color=:RdBu, clims=(-1, 1),
    xrotation=45, size=(450, 400), aspect_ratio=:equal)

Now we construct the full covariance matrix from the correlation matrix and the volatilities: $\boldsymbol{\Sigma} = \mathbf{D}\,\mathbf{C}\,\mathbf{D}$ where $\mathbf{D} = \text{diag}(\sigma_1,\ldots,\sigma_N)$. We verify that the result is symmetric and positive definite — both are required for the quadratic program to be well-posed.

In [ ]:
# construct covariance: Σ = D * C * D -
D = diagm(σ_daily);
Σ_true = D * C_true * D;

# verify properties -
@assert issymmetric(Σ_true) "Covariance matrix must be symmetric"
@assert isposdef(Σ_true) "Covariance matrix must be positive definite"
println("✓ Σ_true is symmetric: $(issymmetric(Σ_true))")
println("✓ Σ_true is positive definite: $(isposdef(Σ_true))")
println("✓ Condition number: $(round(cond(Σ_true), digits=1))")

With the ground-truth parameters in hand, we generate $T = 504$ trading days (≈ 2 years) of synthetic daily returns from a multivariate normal distribution $\mathbf{r}_t \sim \mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\Sigma})$ and save the data for use in the stress-testing example.

In [ ]:
# generate synthetic daily returns -
T = 504; # trading days (≈ 2 years)
dist = MvNormal(μ_true, Σ_true);
returns_matrix = rand(dist, T)'; # T × N matrix

# store in a DataFrame -
df_returns = DataFrame(returns_matrix, asset_names);

# save to data directory -
save(joinpath(_PATH_TO_DATA, "synthetic-returns.jld2"), 
    Dict("returns" => df_returns, "asset_names" => asset_names,
         "mu_true" => μ_true, "Sigma_true" => Σ_true, "C_true" => C_true,
         "sigma_daily" => σ_daily));

# display summary statistics -
println("Generated $(T) days × $(N) assets of synthetic return data.\n")
summary_df = DataFrame(
    "Asset" => asset_names,
    "Mean (daily bps)" => round.(vec(mean(returns_matrix, dims=1)) .* 10000, digits=2),
    "Std (daily bps)" => round.(vec(std(returns_matrix, dims=1)) .* 10000, digits=2),
    "Min (%)" => round.(vec(minimum(returns_matrix, dims=1)) .* 100, digits=3),
    "Max (%)" => round.(vec(maximum(returns_matrix, dims=1)) .* 100, digits=3),
);
pretty_table(summary_df, tf = tf_markdown)

___
## Task 2: Estimate Return and Covariance from the Synthetic Data
In practice, we don't know the true $\boldsymbol{\mu}$ and $\boldsymbol{\Sigma}$ — we estimate them from historical data. Let's compute the sample estimates and measure the estimation error.

> __What are we going to do?__ We'll compute the sample mean $\hat{\boldsymbol{\mu}} = \frac{1}{T}\sum_{t=1}^{T}\mathbf{r}_t$ and sample covariance $\hat{\boldsymbol{\Sigma}} = \frac{1}{T-1}\sum_{t=1}^{T}(\mathbf{r}_t - \hat{\boldsymbol{\mu}})(\mathbf{r}_t - \hat{\boldsymbol{\mu}})^{\top}$, compare them to the true values, visualize the correlation estimation error, and examine the eigenstructure of the covariance matrix.
>
> __What should you see?__ With 504 observations (2 years of daily data), expected return estimates will have noticeable error bars (returns are hard to estimate), while covariance estimates will be more stable. This asymmetry is key to understanding why the optimizer misbehaves.

In [ ]:
# compute sample estimates -
R = Matrix(df_returns); # T × N
μ_hat = vec(mean(R, dims=1));
Σ_hat = cov(R);

# verify symmetry -
@assert isapprox(Σ_hat, Σ_hat', rtol=1e-10) "Sample covariance must be symmetric"

# compare true vs estimated expected returns (annualized) -
comparison = DataFrame(
    "Asset" => asset_names,
    "μ_true (annual %)" => round.(μ_true .* 252 .* 100, digits=2),
    "μ_hat (annual %)" => round.(μ_hat .* 252 .* 100, digits=2),
    "Error (bps)" => round.((μ_hat .- μ_true) .* 252 .* 10000, digits=1)
);

println("True vs. Estimated Expected Returns:")
pretty_table(comparison, tf = tf_markdown)

**Visualize:** Let's compare the true and estimated correlation matrices side by side, and also look at the _estimation error_ matrix $\hat{\mathbf{C}} - \mathbf{C}$.

> __What should you see?__ The estimated correlations should be close to the true values, but with some noise. The error matrix reveals which pairwise correlations are most affected by estimation error — these are the ones the optimizer will exploit most aggressively.

In [ ]:
let
    # compute estimated correlation -
    C_hat = cor(R);
    C_error = C_hat .- C_true;

    # plot three panels -
    p1 = heatmap(C_true, title="True C", xticks=(1:N, asset_names), 
        yticks=(1:N, asset_names), clims=(-1,1), color=:RdBu, xrotation=45);
    p2 = heatmap(C_hat, title="Estimated Ĉ", xticks=(1:N, asset_names), 
        yticks=(1:N, asset_names), clims=(-1,1), color=:RdBu, xrotation=45);
    p3 = heatmap(C_error, title="Error (Ĉ − C)", xticks=(1:N, asset_names), 
        yticks=(1:N, asset_names), clims=(-0.15,0.15), color=:RdBu, xrotation=45);
    
    plot(p1, p2, p3, layout=(1,3), size=(1100,350))
end

**Eigenvalue Analysis:** The eigenvalues of $\hat{\boldsymbol{\Sigma}}$ reveal the _effective dimensionality_ of the risk. If a few eigenvalues dominate, most of the portfolio variance can be explained by a small number of principal components. The condition number $\kappa = \lambda_{\max}/\lambda_{\min}$ measures how sensitive the optimization will be to perturbations.

> __What should you see?__ A high condition number means the optimizer is working in a poorly conditioned space — small changes in inputs produce large changes in the solution. This is one of the mathematical reasons behind the "error maximization" property.

In [ ]:
let
    # compute eigenvalues of estimated covariance -
    λ_true = eigvals(Σ_true) |> sort |> reverse;
    λ_hat = eigvals(Σ_hat) |> sort |> reverse;
    
    # condition numbers -
    κ_true = λ_true[1] / λ_true[end];
    κ_hat = λ_hat[1] / λ_hat[end];
    
    # plot eigenvalue spectrum -
    p = plot(1:N, λ_true .* 10000, marker=:circle, label="True Σ", lw=2, 
        xlabel="Principal Component", ylabel="Eigenvalue (×10⁴)", 
        title="Eigenvalue Spectrum of Covariance Matrix");
    plot!(p, 1:N, λ_hat .* 10000, marker=:diamond, label="Estimated Σ̂", lw=2, 
        linestyle=:dash, size=(600, 400));
    
    println("Condition number (true):      $(round(κ_true, digits=1))")
    println("Condition number (estimated): $(round(κ_hat, digits=1))")
    println("\nVariance explained by PC1: $(round(λ_hat[1]/sum(λ_hat)*100, digits=1))%")
    println("Variance explained by PC1+PC2: $(round(sum(λ_hat[1:2])/sum(λ_hat)*100, digits=1))%")
    
    p
end

___
## Task 3: Solve the Minimum-Variance Problem, Compute the Efficient Frontier, and Test Input Sensitivity
Now we solve the classical minimum-variance problem using the _estimated_ parameters (not the true ones — that's the whole point). We'll then sweep the target return $R$ to trace out the entire efficient frontier, and finally test the "error maximization" property by perturbing expected returns.

> __What are we going to do?__ Three sub-steps: (1) solve for a single target return and inspect the weights, (2) sweep $R$ from 0 to the maximum feasible return to compute the efficient frontier, and (3) perturb the LargeCap expected return by ±100 bps and visualize how weights shift.
>
> __What should you see?__ The optimizer will concentrate weight in the Bond (lowest variance). The efficient frontier will show the risk-return trade-off. The sensitivity test will reveal dramatic weight instability from small input changes.

In [ ]:
# setup the allocation problem -
bounds = hcat(zeros(N), ones(N));  # long-only: 0 ≤ wᵢ ≤ 1
R_target = 0.05 / 252;             # target: 5% annualized

# build and solve -
problem = build(MyPortfolioAllocationProblem;
    μ = μ_hat, Σ = Σ_hat, bounds = bounds, R = R_target);
result = solve_minvariance(problem);

# verify constraints -
@assert isapprox(sum(result.weights), 1.0, rtol=1e-4) "Weights must sum to 1"
@assert all(result.weights .>= -1e-8) "Weights must be non-negative (long-only)"

# display weights -
weights_df = DataFrame(
    "Asset" => asset_names,
    "Weight (%)" => round.(result.weights .* 100, digits=2)
);

println("Minimum-Variance Portfolio Weights (target return = 5% annual):")
println("═"^55)
pretty_table(weights_df, tf = tf_markdown)
println("\nPortfolio expected return (annual): $(round(result.expected_return * 252 * 100, digits=2))%")
println("Portfolio volatility (annual):     $(round(sqrt(result.variance) * sqrt(252) * 100, digits=2))%")

# save for later use -
save(joinpath(_PATH_TO_DATA, "baseline-portfolio.jld2"),
    Dict("weights" => result.weights, "mu_hat" => μ_hat, "Sigma_hat" => Σ_hat,
         "asset_names" => asset_names, "result" => result));

**Visualize:** Portfolio weight allocation as a bar chart.

> __What should you see?__ The Bond asset will likely dominate the allocation because of its much lower variance. This is the _concentration problem_ in action — the optimizer loves low-variance assets regardless of whether that concentration is desirable from a diversification standpoint.

In [ ]:
let
    # Load the baseline portfolio -
    data = load(joinpath(_PATH_TO_DATA, "baseline-portfolio.jld2"));
    weights = data["weights"];
    asset_names = data["asset_names"];

    # Bar chart of weights -
    bar(asset_names, weights .* 100, 
        ylabel="Weight (%)", xlabel="Asset", title="Minimum-Variance Portfolio Weights",
        legend=false, color=:steelblue, bar_width=0.6, size=(600,400),
        ylims=(0, 100))
end

**Efficient Frontier:** Now we sweep the target return $R$ from 0 to the maximum feasible value and solve the quadratic program at each point. This traces out the _efficient frontier_ — the set of lowest-risk portfolios for every attainable return level.

> __What should you see?__ The frontier curves upward and to the right: higher expected return requires accepting more risk. Our baseline portfolio (the red star) sits at one point on this curve. The equal-weight portfolio (green diamond) will typically lie _below_ the frontier — it's suboptimal in the Markowitz sense.

In [ ]:
let
    # sweep target returns -
    R_min = 0.0;
    R_max = maximum(μ_hat) * 0.95;  # just below the max feasible
    n_points = 100;
    R_sweep = range(R_min, stop=R_max, length=n_points) |> collect;
    
    # storage -
    frontier_risk = Float64[];
    frontier_return = Float64[];
    
    for R_i ∈ R_sweep
        try
            prob = build(MyPortfolioAllocationProblem;
                μ = μ_hat, Σ = Σ_hat, bounds = bounds, R = R_i);
            sol = solve_minvariance(prob);
            
            σ_p = sqrt(sol.variance) * sqrt(252) * 100;  # annualized vol %
            μ_p = sol.expected_return * 252 * 100;         # annualized return %
            
            push!(frontier_risk, σ_p);
            push!(frontier_return, μ_p);
        catch
            # infeasible — skip
        end
    end
    
    # compute our baseline portfolio's position -
    baseline_σ = sqrt(result.variance) * sqrt(252) * 100;
    baseline_μ = result.expected_return * 252 * 100;
    
    # compute equal-weight portfolio for comparison -
    w_eq = fill(1.0/N, N);
    eq_var = dot(w_eq, Σ_hat * w_eq);
    eq_ret = dot(μ_hat, w_eq);
    eq_σ = sqrt(eq_var) * sqrt(252) * 100;
    eq_μ = eq_ret * 252 * 100;
    
    # plot -
    p = plot(frontier_risk, frontier_return, lw=2, color=:steelblue, label="Efficient Frontier",
        xlabel="Volatility (annual %)", ylabel="Expected Return (annual %)",
        title="Efficient Frontier", legend=:topleft, size=(700, 450));
    scatter!(p, [baseline_σ], [baseline_μ], marker=:star5, ms=12, color=:red, 
        label="Our Portfolio (R=5%)");
    scatter!(p, [eq_σ], [eq_μ], marker=:diamond, ms=8, color=:green, 
        label="Equal-Weight");
    
    p
end

**Input Sensitivity Test:** Let's test the "error maximization" property directly. We perturb the expected return estimate for LargeCap by ±100 basis points and re-solve each time.

> __What should you see?__ Even modest perturbations to a single asset's expected return can shift weights significantly. This demonstrates that the optimizer is highly sensitive to the _least reliable_ input — estimated expected returns.

In [ ]:
let
    # Load data -
    data = load(joinpath(_PATH_TO_DATA, "baseline-portfolio.jld2"));
    μ_hat = data["mu_hat"];
    Σ_hat = data["Sigma_hat"];
    asset_names = data["asset_names"];
    baseline_weights = data["weights"];
    N = length(asset_names);
    
    # Setup -
    bounds = hcat(zeros(N), ones(N));
    R_target = 0.05 / 252;
    
    # Perturb LargeCap (index 1) expected return by ±100 bps annually -
    perturbations_bps = [-100, -50, 0, 50, 100];
    results = DataFrame();

    for δ_bps in perturbations_bps
        
        # perturb -
        μ_perturbed = copy(μ_hat);
        μ_perturbed[1] += (δ_bps / 10000) / 252; # convert annual bps to daily

        # solve -
        problem = build(MyPortfolioAllocationProblem;
            μ = μ_perturbed, Σ = Σ_hat, bounds = bounds, R = R_target);
        result = solve_minvariance(problem);

        # store -
        row = DataFrame("Perturbation (bps)" => δ_bps, 
            [name => round(w * 100, digits=2) for (name, w) in zip(asset_names, result.weights)]...);
        results = vcat(results, row);
    end

    println("LargeCap expected return perturbation → weight sensitivity:")
    pretty_table(results, tf=tf_markdown)
end

**Visualize:** Let's plot the weight sensitivity as a grouped bar chart — each cluster of bars shows how one asset's weight changes across the perturbation levels.

In [ ]:
let
    # recompute sensitivity results for plotting -
    perturbations_bps = [-100, -50, 0, 50, 100];
    weight_matrix = zeros(length(perturbations_bps), N);
    
    for (j, δ_bps) ∈ enumerate(perturbations_bps)
        μ_perturbed = copy(μ_hat);
        μ_perturbed[1] += (δ_bps / 10000) / 252;
        
        prob = build(MyPortfolioAllocationProblem;
            μ = μ_perturbed, Σ = Σ_hat, bounds = bounds, R = R_target);
        sol = solve_minvariance(prob);
        weight_matrix[j, :] = sol.weights .* 100;
    end
    
    # grouped bar chart -
    groupedbar(asset_names, weight_matrix', 
        bar_position=:dodge, bar_width=0.15,
        label=permutedims(["$(δ) bps" for δ ∈ perturbations_bps]),
        ylabel="Weight (%)", xlabel="Asset",
        title="Weight Sensitivity to LargeCap μ Perturbation",
        size=(750, 450), legend=:topright)
end

___
## Summary

> __Key Takeaways:__
>
> * **Concentration**: The minimum-variance optimizer heavily loads the Bond asset (lowest variance), producing a portfolio that is "optimal" on paper but poorly diversified in practice
> * **Efficient Frontier**: Sweeping the target return traces out the full risk-return trade-off. Our baseline portfolio sits on this curve; the equal-weight portfolio typically lies below it (suboptimal in the Markowitz sense, but more robust)
> * **Input Sensitivity**: Perturbing LargeCap's expected return by just ±100 basis points produces dramatic weight shifts — confirming Michaud's "error maximization" diagnosis
> * **Eigenstructure**: The condition number of $\hat{\boldsymbol{\Sigma}}$ quantifies how sensitive the optimizer is to perturbations. High condition numbers predict unstable solutions

In the [next example](eCornell-AI-Finance-L1-Example-StressTestMinVariancePortfolio-May-2026.ipynb), we'll stress-test this baseline portfolio under correlation shifts, price drops, and higher trading costs.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The examples use synthetic data and simplified models.